# Exploratory Data Analysis

Subject: Titanic Survival Prediction

## Data Cleaning and Preprocessing

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


In [ ]:
# Create the dataframe
datanic_origin = pd.read_csv("data/train.csv")
datanic_origin.head()


In [ ]:
# Copy the dataframe
datanic = datanic_origin.copy()


In [ ]:
# Shape and info of the dataframe
print(f"Shape of the database: \n {datanic.shape}\n")
print(f"Column names: \n {datanic.columns}\n")


In [ ]:
print(f"Shape of the database: \n {datanic.info()}")


In [ ]:
def incomplete(data):
    total_incomplete = data[:].isnull().sum()
    return total_incomplete


datanic.apply(incomplete, axis=0)


In [ ]:
# Drop columns with too many missing values and non-relevant columns like PassengerId and Name
# Drop empty rows

datanic.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1, inplace=True)
datanic.dropna(axis=0, subset=["Age", "Embarked"], inplace=True)
datanic.apply(incomplete, axis=0)


In [ ]:
# Create a column for age distribution classified as "Minor, Adult, Senior"
def sort_age(line):
    age_label = line["Age"]
    if line["Age"] < 18:
        return "Minor"
    elif line["Age"] >= 18:
        return "Adult"
    else:
        return "Unknown"


datanic["Age_label"] = datanic.apply(sort_age, axis=1)
datanic.loc[:, ["Pclass", "Sex", "Age", "Age_label", "Survived"]].head()


In [ ]:
datanic["Age_Sex"] = datanic.loc[:, "Age_label"] + " " + datanic.loc[:, "Sex"]
datanic.loc[:, ["Pclass", "Sex", "Age_Sex", "Age", "Age_label", "Survived"]].head()


Creating two dataframes with fewer columns and data to reduce execution time when generating graphs with seaborn

In [ ]:
datanic_small = datanic.loc[:, ["Pclass", "Sex", "Age_label", "Age_Sex", "Survived"]]
datanic_small


In [ ]:
datanic_medium = datanic.loc[
    :, ["Embarked", "Pclass", "Sex", "Age_Sex", "Age", "Age_label", "Survived"]
]
datanic_medium


### 1.b. Data Exploration: Analysis and Visualizations

Observation of distribution by categories and correlation between categories

In [ ]:
print(
    f"Distribution of people by class: \n{datanic.loc[:, 'Pclass'].value_counts()}\n"
)
print(
    f"Distribution of people by gender: \n{datanic.loc[:, 'Sex'].value_counts()}\n"
)
print(
    f"Distribution of people by age (adult/minor): \n{datanic.loc[:, 'Age_label'].value_counts()}\n"
)
print(
    f"Number of people deceased (0) and survivors (1): \n{datanic.loc[:, 'Survived'].value_counts()}\n"
)
print(
    f"Distribution of people by embarkation port: \n{datanic.loc[:, 'Embarked'].value_counts()}"
)


In [ ]:
# Correlation matrix between numeric variables of the dataframe
plt.show(sns.heatmap(datanic.corr(), vmin=-1, vmax=+1, annot=True, cmap="coolwarm"))


#### Does class and gender influence survival chances?

In [ ]:
# Survival chances by class

datanic.pivot_table(index="Pclass", values=["Survived"], aggfunc=[np.mean, sum])


In [ ]:
# Survival chances by gender (non-binary excluded) and total

datanic.pivot_table(index=["Sex"], values=["Survived"], aggfunc=[np.mean, sum])


In [ ]:
# Survival chances by class and gender and total

datanic.pivot_table(
    index=["Pclass", "Sex"], values=["Survived"], aggfunc=[np.mean, sum]
)


In [ ]:
sns.barplot(
    data=datanic_small,
    x="Sex",
    y="Survived",
    hue="Pclass",
    palette="colorblind",
    estimator="mean",
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_small,
    x="Sex",
    y="Survived",
    hue="Pclass",
    palette="colorblind",
    estimator="sum",
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_small,
    x="Pclass",
    y="Survived",
    hue="Sex",
    palette="colorblind",
    estimator="mean",
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_small,
    x="Pclass",
    y="Survived",
    hue="Sex",
    palette="colorblind",
    estimator="sum",
    errwidth=0,
)
plt.show()


Both class and gender appear to be highly discriminating factors in measuring passenger survival rates.
If we ranked people by survival likelihood, the results speak for themselves:
1. Woman in 1st class
2. Woman in 2nd class
3. Woman in 3rd class
4. Man in 1st class
5. Man in 2nd class
6. Man in 3rd class

We will explore the rest of the data to see if there are other discriminating factors in the survival rate.

#### Does the port of embarkation influence survival chances?

In [ ]:
# Survival by embarkation port

datanic.pivot_table(index=["Embarked"], values=["Survived"], aggfunc=[np.mean, sum])


In [ ]:
datanic.loc[:, "Embarked"].value_counts()


In [ ]:
datanic.groupby("Embarked").Survived.value_counts()


In [ ]:
datanic.groupby("Embarked").Pclass.value_counts()


Several interesting data points should be noted:
- The embarkation port "Q" for Queenstown in Ireland was primarily used by passengers traveling in 3rd class. Only 2 people were in 1st, 2 in 2nd class versus 24 in 3rd.
- The embarkation port "C" for Cherbourg in France was primarily used by passengers traveling in 1st class: 74 people versus only 15 for 2nd class and 41 for 3rd class. The "C" category is thus more represented by 1st class passengers while for "Q" and "S" the 1st class is often the category with the fewest passengers.

In [ ]:
# Survival by embarkation port and by class
datanic.pivot_table(
    index=["Embarked", "Pclass"], values=["Survived"], aggfunc=[np.mean, sum]
)


In [ ]:
# Survival by embarkation port and by gender
datanic.pivot_table(
    index=["Embarked", "Sex"], values=["Survived"], aggfunc=[np.mean, sum]
)


For "Q":
- 1st and 2nd class, only 2 people per class and a 50% survival rate per class. The average is not relevant for so few data points.

For "C":
- The 3rd class survival rate is 43%, higher than "Q" and "S", which are 25% and 21% respectively. It remains to be seen if this is justified by a larger proportion of women in the "C" group?

In [ ]:
datanic.groupby("Embarked").Sex.value_counts()


The distribution by gender outside of class is fairly homogeneous for "Q" and "C", but heterogeneous for "S". There are far more men than women, which may help explain the lower survival rate for those embarking at Southampton compared to Cherbourg or Queenstown.

In [ ]:
# Survival by embarkation port, by class and by gender

datanic.pivot_table(
    index=["Embarked", "Pclass", "Sex"], values=["Survived"], aggfunc=[np.mean, sum]
)


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Embarked",
    y="Survived",
    hue="Sex",
    palette="colorblind",
    estimator="mean",
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Embarked",
    y="Survived",
    hue="Sex",
    palette="colorblind",
    estimator="sum",
    errwidth=0,
)
plt.show()


We find the same pattern: women survived more than men, regardless of embarkation port.
The difference between the 3 categories lies more in the class distribution:
- There are more 1st class passengers for Cherbourg
- There are almost exclusively 3rd class passengers for Queenstown.

In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Embarked",
    y="Survived",
    hue="Pclass",
    palette="colorblind",
    estimator="mean",
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Embarked",
    y="Survived",
    hue="Pclass",
    palette="colorblind",
    estimator="sum",
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Embarked",
    y="Survived",
    hue="Age_label",
    palette="colorblind",
    estimator="mean",
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Embarked",
    y="Survived",
    hue="Age_label",
    palette="colorblind",
    estimator="sum",
    errwidth=0,
)
plt.show()


The survival rate thus appears to be more linked to two criteria:
- the class of the passenger
- the gender

Are there other discriminating factors?

#### Does age and the adult/minor distinction influence survival chances?

In [ ]:
datanic.pivot_table(index=["Age_label"], values=["Survived"], aggfunc=[np.mean, sum])


In [ ]:
datanic.pivot_table(
    index=["Age_label", "Sex"], values=["Survived"], aggfunc=[np.mean, sum]
)


In [ ]:
datanic.pivot_table(
    index=["Age_label", "Sex", "Pclass"], values=["Survived"], aggfunc=[np.mean, sum]
)


In [ ]:
# Distribution by age of passengers
vis1 = sns.displot(datanic["Age"], bins=16, color="forestgreen")


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Age_label",
    y="Survived",
    hue="Sex",
    palette="colorblind",
    estimator="mean",
    order=["Minor", "Adult"],
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Age_label",
    y="Survived",
    hue="Sex",
    palette="colorblind",
    estimator="sum",
    order=["Minor", "Adult"],
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Age_label",
    y="Survived",
    hue="Pclass",
    palette="colorblind",
    estimator="mean",
    order=["Minor", "Adult"],
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Age_label",
    y="Survived",
    hue="Pclass",
    palette="colorblind",
    estimator="sum",
    order=["Minor", "Adult"],
    errwidth=0,
)
plt.show()


In [ ]:
sns.barplot(
    data=datanic_medium,
    x="Age_Sex",
    y="Survived",
    hue="Pclass",
    palette="colorblind",
    order=["Minor male", "Minor female", "Adult male", "Adult female"],
    errwidth=0,
)
plt.xticks(rotation=70)
plt.show()


In [ ]:
sns.countplot(
    data=datanic_small,
    x="Age_Sex",
    hue="Pclass",
    palette="colorblind",
    order=["Minor male", "Minor female", "Adult male", "Adult female"],
)
plt.show()


The minor/adult criterion reduces the gap in survival rate between boys and girls, but the survival rate is still higher for girls (~70%) than for boys (~39%). Minors have a higher survival rate than adults in any case. However, note that the sample size for minors is smaller than for adults.

This criterion therefore remains discriminatory and can be added to class and gender to study a passenger's survival rate.

### Do survival chances increase when paying a higher ticket price?

In [ ]:
datanic.pivot_table(index=["Pclass"], values=["Fare", "Survived"], aggfunc=[np.mean])


In [ ]:
datanic.pivot_table(
    index=["Pclass", "Sex"], values=["Fare", "Survived"], aggfunc=[np.mean]
)


In [ ]:
datanic.pivot_table(
    index=["Pclass", "Age_Sex"], values=["Fare", "Survived"], aggfunc=[np.mean]
)


In [ ]:
sns.barplot(
    data=datanic,
    x="Age_Sex",
    y="Fare",
    hue="Pclass",
    palette="colorblind",
    order=["Minor male", "Minor female", "Adult male", "Adult female"],
    errwidth=0,
)
plt.xticks(rotation=70)
plt.show()


In [ ]:
sns.displot(datanic["Fare"], bins=50, color="forestgreen")
plt.show()


In [ ]:
sns.scatterplot(
    data=datanic, x="Age_Sex", y="Fare", hue="Survived", palette="colorblind"
)
plt.xticks(rotation=70)
plt.show()


In [ ]:
sns.barplot(
    data=datanic, x="Pclass", y="Fare", hue="Survived", palette="colorblind", errwidth=0
)
plt.show()


A high ticket price appears to increase survival chances but may be linked to the departure port and passenger class.